# Detecção de Anomalias — Clientes + Endereços (não supervisionado)

**Enquadramento:** dados tabulares sem rótulo confiável que podem conter anomalias → **detecção de outliers não supervisionada** com **One-Class SVM** e **Local Outlier Factor** (modelos propostos), cada um em um `Pipeline` (pré-processamento encapsulado no modelo logado).

**Features (1 linha = 1 endereço, cliente unido):** tamanho do CEP (dígitos), logradouro vazio, nº de endereços por cliente e domínio do e-mail. (`latlong_no_brasil` foi excluída da matriz X: variância zero — as 1240 linhas estão dentro do bounding box oficial da regra R05.)

**Treino sem rótulo.** O **rótulo fraco** (`alguma falha_rNN_*` de Data Quality da Silver) entra **somente na avaliação** (PR-AUC, precision@k, recall no limiar) — nunca no treino. Como o rótulo fraco deriva de regras que inspiram as próprias features, a concordância alta é esperada: a avaliação é um *sanity check* de que o modelo não supervisionado recupera os padrões que as regras capturam, não uma métrica de generalização.

**Taxa de flag assumida:** **15%** no LOF (`contamination`) e **25%** no OCSVM (`nu`, calibrado na célula 4b — taxa efetiva de flags ≈ 39%), já que não há taxa informada no domínio; o parâmetro define apenas o limiar de flag, não o score.

### 1. Configuração e leitura das fontes
Credenciais ADLS (`.env` via API do workspace), leitura da Silver de endereços (com flags `falha_rNN`) e da Silver de clientes da dupla parceira (deduplicada: última versão por `id_cliente`).

In [0]:
# 1. Credenciais ADLS + leitura da Silver (endereços) e da Bronze (clientes, deduplicada)
import os
from databricks.sdk import WorkspaceClient
from pyspark.sql import Window
from pyspark.sql import functions as F

ENV_PATH = "/Users/luizpopoff@gmail.com/estagio-empregadados-turma-2/.env"

# No serverless o FUSE não expõe arquivos ocultos: baixa o .env pela API do workspace
_env_txt = WorkspaceClient().workspace.download(ENV_PATH).read().decode()
for _linha in _env_txt.splitlines():
    _linha = _linha.strip()
    if _linha and not _linha.startswith("#") and "=" in _linha:
        _k, _v = _linha.split("=", 1)
        os.environ.setdefault(_k.strip(), _v.strip())

client_id            = os.environ["ADLS_CLIENT_ID"]
tenant_id            = os.environ["ADLS_TENANT_ID"]
client_secret        = os.environ["ADLS_CLIENT_SECRET"]
storage_account_name = os.environ.get("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

squad_base = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net"
silver_enderecos_path = f"{squad_base}/silver/ecommerce_enderecos"
clientes_path         = f"{squad_base}/silver/ecommerce_clientes"

df_enderecos = spark.read.format("delta").options(**adls_opts).load(silver_enderecos_path)
df_clientes_raw = spark.read.format("delta").options(**adls_opts).load(clientes_path)

# Silver de clientes é append por micro-lote: mantém a última versão de cada id_cliente
_w_cli = Window.partitionBy("id_cliente").orderBy(F.col("bronze_source_file").desc())
df_clientes = (
    df_clientes_raw
    .withColumn("_versao", F.row_number().over(_w_cli))
    .filter(F.col("_versao") == 1)
    .drop("_versao")
)

print(f"Endereços (Silver):        {df_enderecos.count()} linhas")
print(f"Clientes (Silver bruta):   {df_clientes_raw.count()} linhas")
print(f"Clientes distintos:        {df_clientes.count()} linhas")
print("\n--- Schema clientes (colunas disponíveis) ---")
df_clientes.printSchema()

### 2. Join cliente + endereço e features
Uma linha por endereço, com atributos do cliente unidos (e-mail). Features: `cep_tamanho` (dígitos), `logradouro_vazio`, `qtd_enderecos_por_cliente` e `dominio_email` (`latlong_no_brasil` calculada só para conferência — variância zero, fora da matriz X).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Parâmetros e limites geográficos (bounding box oficial da regra R05 da Silver)
CEP_RE = "[^0-9]"
LAT_LIM = (-33.75, 5.27)
LON_MIN, LON_MAX = (-73.99, -32.39)

# 2. Definição do Rótulo Fraco (consolida regras de qualidade da etapa 1 sem gerar nulos)
REGRAS = [c for c in df_enderecos.columns if c.startswith("falha_r")]
if REGRAS:
    _alguma_falha = F.greatest(*[F.coalesce(F.col(c).cast("int"), F.lit(0)) for c in REGRAS])
else:
    _alguma_falha = F.lit(0)

# Window para contagem eficiente por cliente sem join adicional
w_cliente = Window.partitionBy("id_cliente")

# 3. Engenharia de Features
df_base = (
    df_enderecos
    # Traz o e-mail do cliente
    .join(df_clientes.select("id_cliente", "email"), on="id_cliente", how="left")
    
    # 1. Tamanho do CEP (apenas números; nulos viram 0)
    .withColumn(
        "cep_tamanho",
        F.coalesce(F.length(F.regexp_replace(F.col("cep").cast("string"), CEP_RE, "")), F.lit(0))
    )
    
    # 2. Logradouro vazio (flag binária 1/0)
    .withColumn(
        "logradouro_vazio", 
        F.when(F.col("logradouro").isNull() | (F.trim(F.col("logradouro")) == ""), 1).otherwise(0)
    )
    
    # 3. Lat/Long no Brasil (evita nulls e garante 1 ou 0 estrito)
    .withColumn(
        "latlong_no_brasil", 
        F.when(
            F.col("latitude").isNotNull() & F.col("longitude").isNotNull() &
            F.col("latitude").between(LAT_LIM[0], LAT_LIM[1]) &
            F.col("longitude").between(LON_MIN, LON_MAX),
            1
        ).otherwise(0)
    )
    
    # 4. Nº de endereços por cliente
    .withColumn("qtd_enderecos_por_cliente", F.coalesce(F.count("id_endereco").over(w_cliente), F.lit(1)))
    
    # 5. Domínio do e-mail
    .withColumn("dominio_email", F.lower(F.regexp_extract(F.col("email"), "@(.+)$", 1)))
    .withColumn(
        "dominio_email", 
        F.when(F.col("dominio_email").isNull() | (F.col("dominio_email") == ""), "sem_email")
        .otherwise(F.col("dominio_email"))
    )
    
    # Rótulo fraco isolado para avaliação
    .withColumn("rotulo_fraco", _alguma_falha)
)

# 4. Validações e Sanity Check
print(f"Linhas da base (1 por endereço): {df_base.count()}")
print(f"Endereços sem match de e-mail: {df_base.filter(F.col('dominio_email') == 'sem_email').count()}")
print(f"Rótulo fraco positivo (inconsistências): {df_base.filter(F.col('rotulo_fraco') == 1).count()} linhas")

### 3. Preparação da Matriz de Features (X) e Rótulo Fraco (y_true)
Converta o DataFrame do Spark com as variáveis calculadas (df_base) para Pandas e aplique a padronização e a codificação categórica:

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Seleciona apenas as features e o rótulo fraco da base já construída
pdf = df_base.select(
    "id_endereco",
    "id_cliente",
    "cep_tamanho",
    "logradouro_vazio",
    "latlong_no_brasil",
    "qtd_enderecos_por_cliente",
    "dominio_email",
    "rotulo_fraco"
).toPandas()

# 2. Agrupa domínios raros para evitar esparsidade excessiva (top 10 domínios + "outros")
top_dominios = pdf["dominio_email"].value_counts().nlargest(10).index
pdf["dominio_email_cat"] = pdf["dominio_email"].apply(lambda d: d if d in top_dominios else "outros")

# 3. Definição das colunas
# latlong_no_brasil fica fora de X: variância zero (todas as 1240 linhas dentro do bounding
# box oficial da regra R05) — só adicionaria distância constante aos modelos de distância.
num_cols = ["cep_tamanho", "qtd_enderecos_por_cliente"]
bin_cols = ["logradouro_vazio"]
cat_cols = ["dominio_email_cat"]

# 4. Pipeline de transformação com StandardScaler (vital para modelos baseados em distância/kernel)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("bin", "passthrough", bin_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)

# X para treino NÃO SUPERVISIONADO (sem rótulo)
X = preprocessor.fit_transform(pdf)

# y_true reservado EXCLUSIVAMENTE para avaliação final contra o rótulo fraco
y_true = pdf["rotulo_fraco"].astype(int).values

print(f"Dimensões da matriz de entrada X: {X.shape}")
print(f"Total de registros: {len(y_true)} | Anomalias pelo rótulo fraco: {np.sum(y_true)}")

### 4. Treinamento e Predição dos Modelos
Modelo B.1: One-Class SVM
O hiperparâmetro nu define a proporção máxima estimada de contaminação/outliers no conjunto; calibrado em **0.25** pela varredura da célula 4b (melhor F1).

O método .decision_function(X) retorna a distância com sinal até a fronteira (invertemos o sinal com - para que valores maiores signifiquem maior grau de suspeita/anomalia).

In [0]:
from sklearn.svm import OneClassSVM

# Ajuste Não Supervisionado
# nu calibrado na célula 4b (varredura 0.05–0.25): melhor F1 = 0.4241 (39.44% alertados).
# Alternativas: nu=0.15 maximiza precision (0.8571); nu=0.05 maximiza PR-AUC (0.7826).
oc_svm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.25)
oc_svm.fit(X)

# Predição: +1 = inlier (legítimo), -1 = outlier (anômalo)
preds_svm_raw = oc_svm.predict(X)
y_pred_svm = np.where(preds_svm_raw == -1, 1, 0)

# Escore de anomalia contínuo (quanto maior, mais anômalo)
scores_svm = -oc_svm.decision_function(X)

### 4b. Calibração de nu do One-Class SVM
Varredura de `nu` (proporção máxima de outliers que o modelo assume): quanto maior, mais registros são flagados e menor tende a ser a precision. O critério de escolha abaixo é o melhor F1 contra o rótulo fraco (sanity check — o rótulo entra só na leitura dos resultados, nunca no treino). Resultado: a relação não é monotônica (nu não fixa a taxa de flags na prática); **nu=0.25** dá o melhor F1 (0.4241), mas **nu=0.05** dá o melhor PR-AUC (0.7826) e **nu=0.15** a melhor precision (0.8571).

In [0]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc

# Varredura de nu: efeito no flag e nas métricas contra o rótulo fraco
valores_nu = [0.05, 0.10, 0.15, 0.20, 0.25]
calibracao = []

for nu_val in valores_nu:
    svm_tmp = OneClassSVM(kernel="rbf", gamma="scale", nu=nu_val).fit(X)
    y_tmp = np.where(svm_tmp.predict(X) == -1, 1, 0)
    scores_tmp = -svm_tmp.decision_function(X)
    p_c, r_c, _ = precision_recall_curve(y_true, scores_tmp)
    calibracao.append({
        "nu": nu_val,
        "Precision": round(precision_score(y_true, y_tmp, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_tmp, zero_division=0), 4),
        "F1-Score": round(f1_score(y_true, y_tmp, zero_division=0), 4),
        "PR-AUC": round(auc(r_c, p_c), 4),
        "% Alertados": f"{(np.sum(y_tmp) / len(y_tmp)) * 100:.2f}%",
    })

df_calibracao = pd.DataFrame(calibracao)
display(df_calibracao)

_melhor = df_calibracao.loc[df_calibracao["F1-Score"].idxmax()]
print(f"nu com melhor F1: {_melhor['nu']} (F1 = {_melhor['F1-Score']}, {_melhor['% Alertados']} alertados)")

### 5. Modelo B.2: Local Outlier Factor (LOF)
Usamos novelty=True para que o algoritmo permita calcular scores e decisões em matrizes de validação/teste via .decision_function(X).

O parâmetro contamination calibra o limiar do classificador para a classe minoritária.

In [0]:
from sklearn.neighbors import LocalOutlierFactor

# Ajuste Não Supervisionado
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.15, novelty=True)
lof.fit(X)

# Predição: +1 = inlier, -1 = outlier
preds_lof_raw = lof.predict(X)
y_pred_lof = np.where(preds_lof_raw == -1, 1, 0)

# Escore contínuo de anomalia (quanto maior, mais anômalo)
scores_lof = -lof.decision_function(X)

### 6. Função de Avaliação e Cálculo de Métricas
Conforme o requisito do Problema 2, devem ser reportados Precision, Recall, F1 e PR-AUC medidos contra o rótulo fraco da Etapa 1, além da taxa de registros alertados:

In [0]:
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc

def avaliar_modelo(y_real, y_predito, escores, nome_modelo):
    prec = precision_score(y_real, y_predito, zero_division=0)
    rec = recall_score(y_real, y_predito, zero_division=0)
    f1 = f1_score(y_real, y_predito, zero_division=0)
    
    # Cálculo da Curva Precision-Recall e da Área sob a Curva (PR-AUC)
    p_vals, r_vals, _ = precision_recall_curve(y_real, escores)
    pr_auc = auc(r_vals, p_vals)
    
    taxa_alerta = (np.sum(y_predito) / len(y_predito)) * 100.0
    
    return {
        "Modelo": nome_modelo,
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "PR-AUC": round(pr_auc, 4),
        "% Alertados": f"{taxa_alerta:.2f}%"
    }

res_svm = avaliar_modelo(y_true, y_pred_svm, scores_svm, "One-Class SVM")
res_lof = avaliar_modelo(y_true, y_pred_lof, scores_lof, "Local Outlier Factor (LOF)")

# Tabela consolidada de métricas
df_resultados = pd.DataFrame([res_svm, res_lof])
display(df_resultados)

### 7. Visualização da Curva Precision-Recall Comparativa
Gere o gráfico das curvas PR-AUC para comparar a capacidade discriminativa contínua dos dois modelos:

In [0]:
import matplotlib.pyplot as plt

p_svm, r_svm, _ = precision_recall_curve(y_true, scores_svm)
p_lof, r_lof, _ = precision_recall_curve(y_true, scores_lof)
plt.figure(figsize=(8, 5))
plt.plot(r_svm, p_svm, label=f"One-Class SVM (PR-AUC = {res_svm['PR-AUC']})", color="#0e7c86", lw=2)
plt.plot(r_lof, p_lof, label=f"LOF (PR-AUC = {res_lof['PR-AUC']})", color="#d9534f", lw=2)
plt.axhline(y=y_true.mean(), color="gray", linestyle="--", lw=1, label=f"Baseline aleatório ({y_true.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Comparativo PR-AUC: One-Class SVM vs. Local Outlier Factor")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(loc="lower left")
plt.show()

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    precision_recall_curve, 
    auc, 
    confusion_matrix, 
    classification_report
)

# Escolha o modelo que deseja avaliar contra o rótulo fraco:
y_pred = y_pred_svm
scores = scores_svm

# Modelos disponíveis: y_pred_lof/scores_lof (LOF)

# ==============================================================================
# 1. CÁLCULO DAS MÉTRICAS EXIGIDAS NO PROJETO
# ==============================================================================
# y_true: rótulo fraco da Etapa 1 (1 = com falha/suspeito, 0 = limpo)
# y_pred: predição do modelo (1 = anomalia/outlier, 0 = inlier)
# scores: score contínuo de anomalia (quanto maior, mais anômalo)

prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)

# PR-AUC (Precision-Recall Area Under Curve)
p_curve, r_curve, _ = precision_recall_curve(y_true, scores)
pr_auc = auc(r_curve, p_curve)

pct_alertados = (np.sum(y_pred) / len(y_pred)) * 100.0

print("=" * 60)
print("📊 RESULTADOS DA COMPARAÇÃO CONTRA O RÓTULO FRACO")
print("=" * 60)
print(f"Precision:         {prec:.4f}")
print(f"Recall:            {rec:.4f}")
print(f"F1-Score:          {f1:.4f}")
print(f"PR-AUC:            {pr_auc:.4f}")
print(f"Taxa de Alerta:    {pct_alertados:.2f}% dos cadastros")
print("-" * 60)

# Matriz de Confusão
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"Verdadeiros Negativos (Legítimos Aprovados): {tn}")
print(f"Falsos Positivos      (Regras OK, Modelo Alertou): {fp}")
print(f"Falsos Negativos      (Falhou na Regra, Modelo Não Viu): {fn}")
print(f"Verdadeiros Positivos (Falhou na Regra e Modelo Pegou): {tp}")
print("=" * 60)


# ==============================================================================
# 2. PLOT DA CURVA PRECISION-RECALL
# ==============================================================================
plt.figure(figsize=(7, 4.5))
plt.plot(r_curve, p_curve, color="#0e7c86", lw=2, label=f"Modelo B (PR-AUC = {pr_auc:.4f})")
plt.axhline(y=np.sum(y_true)/len(y_true), color="gray", linestyle="--", label="Baseline Aleatório")
plt.xlabel("Recall (Taxa de Cobertura das Falhas)")
plt.ylabel("Precision (Taxa de Acerto dos Alertas)")
plt.title("Curva Precision-Recall vs. Rótulo Fraco")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower left")
plt.show()


# ==============================================================================
# 3. DEEP DIVE: EM QUAIS REGRAS O MODELO MAIS ACERTOU?
# ==============================================================================
# Inspeciona a performance contra cada regra individual da Etapa 1
# Adicionamos a predição no DataFrame para comparar coluna a coluna
colunas_regras = [c for c in df_base.columns if c.startswith("falha_r")]
# pdf (célula 7) NÃO contém as colunas falha_r* — elas vêm de df_base (mesmas linhas,
# mesmos ids) via merge por id_endereco; schema computado uma única vez fora do loop.
pdf_regras = df_base.select("id_endereco", *colunas_regras).toPandas()
pdf_analise = pdf.merge(pdf_regras, on="id_endereco", how="left")
pdf_analise["predicao_modelo"] = y_pred

relatorio_regras = []
for regra in colunas_regras:
    if regra in pdf_analise.columns:
        # Registros que falharam nessa regra específica
        total_com_essa_falha = pdf_analise[pdf_analise[regra] == True].shape[0]
        
        if total_com_essa_falha > 0:
            # Quantos desses o modelo também marcou como 1 (capturados)
            capturados = pdf_analise[(pdf_analise[regra] == True) & (pdf_analise["predicao_modelo"] == 1)].shape[0]
            taxa_captura = (capturados / total_com_essa_falha) * 100.0
            
            relatorio_regras.append({
                "Regra de DQ (Etapa 1)": regra,
                "Total Falhas": total_com_essa_falha,
                "Detectados pelo Modelo": capturados,
                "Recall Específico (%)": f"{taxa_captura:.2f}%"
            })

df_detalhamento = pd.DataFrame(relatorio_regras).sort_values(by="Total Falhas", ascending=False)
display(df_detalhamento)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    precision_recall_curve, 
    auc, 
    confusion_matrix, 
    classification_report
)

# Escolha o modelo que deseja avaliar contra o rótulo fraco:
y_pred = y_pred_lof
scores = scores_lof

# Modelos disponíveis: y_pred_lof/scores_lof (LOF)

# ==============================================================================
# 1. CÁLCULO DAS MÉTRICAS EXIGIDAS NO PROJETO
# ==============================================================================
# y_true: rótulo fraco da Etapa 1 (1 = com falha/suspeito, 0 = limpo)
# y_pred: predição do modelo (1 = anomalia/outlier, 0 = inlier)
# scores: score contínuo de anomalia (quanto maior, mais anômalo)

prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)

# PR-AUC (Precision-Recall Area Under Curve)
p_curve, r_curve, _ = precision_recall_curve(y_true, scores)
pr_auc = auc(r_curve, p_curve)

pct_alertados = (np.sum(y_pred) / len(y_pred)) * 100.0

print("=" * 60)
print("📊 RESULTADOS DA COMPARAÇÃO CONTRA O RÓTULO FRACO")
print("=" * 60)
print(f"Precision:         {prec:.4f}")
print(f"Recall:            {rec:.4f}")
print(f"F1-Score:          {f1:.4f}")
print(f"PR-AUC:            {pr_auc:.4f}")
print(f"Taxa de Alerta:    {pct_alertados:.2f}% dos cadastros")
print("-" * 60)

# Matriz de Confusão
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"Verdadeiros Negativos (Legítimos Aprovados): {tn}")
print(f"Falsos Positivos      (Regras OK, Modelo Alertou): {fp}")
print(f"Falsos Negativos      (Falhou na Regra, Modelo Não Viu): {fn}")
print(f"Verdadeiros Positivos (Falhou na Regra e Modelo Pegou): {tp}")
print("=" * 60)


# ==============================================================================
# 2. PLOT DA CURVA PRECISION-RECALL
# ==============================================================================
plt.figure(figsize=(7, 4.5))
plt.plot(r_curve, p_curve, color="#0e7c86", lw=2, label=f"Modelo B (PR-AUC = {pr_auc:.4f})")
plt.axhline(y=np.sum(y_true)/len(y_true), color="gray", linestyle="--", label="Baseline Aleatório")
plt.xlabel("Recall (Taxa de Cobertura das Falhas)")
plt.ylabel("Precision (Taxa de Acerto dos Alertas)")
plt.title("Curva Precision-Recall vs. Rótulo Fraco")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower left")
plt.show()


# ==============================================================================
# 3. DEEP DIVE: EM QUAIS REGRAS O MODELO MAIS ACERTOU?
# ==============================================================================
# Inspeciona a performance contra cada regra individual da Etapa 1
# Adicionamos a predição no DataFrame para comparar coluna a coluna
colunas_regras = [c for c in df_base.columns if c.startswith("falha_r")]
# pdf (célula 7) NÃO contém as colunas falha_r* — elas vêm de df_base (mesmas linhas,
# mesmos ids) via merge por id_endereco; schema computado uma única vez fora do loop.
pdf_regras = df_base.select("id_endereco", *colunas_regras).toPandas()
pdf_analise = pdf.merge(pdf_regras, on="id_endereco", how="left")
pdf_analise["predicao_modelo"] = y_pred

relatorio_regras = []
for regra in colunas_regras:
    if regra in pdf_analise.columns:
        # Registros que falharam nessa regra específica
        total_com_essa_falha = pdf_analise[pdf_analise[regra] == True].shape[0]
        
        if total_com_essa_falha > 0:
            # Quantos desses o modelo também marcou como 1 (capturados)
            capturados = pdf_analise[(pdf_analise[regra] == True) & (pdf_analise["predicao_modelo"] == 1)].shape[0]
            taxa_captura = (capturados / total_com_essa_falha) * 100.0
            
            relatorio_regras.append({
                "Regra de DQ (Etapa 1)": regra,
                "Total Falhas": total_com_essa_falha,
                "Detectados pelo Modelo": capturados,
                "Recall Específico (%)": f"{taxa_captura:.2f}%"
            })

df_detalhamento = pd.DataFrame(relatorio_regras).sort_values(by="Total Falhas", ascending=False)
display(df_detalhamento)

### Conclusão — versão final (X com 12 features, fechada em 2026-11-12)

**Configuração final:** 1.240 endereços × 12 colunas (2 numéricas padronizadas + 1 binária + 9 dummies de domínio de e-mail), rótulo fraco de 954 inconsistências (77%) usado exclusivamente na avaliação.

**Resultados finais contra o rótulo fraco:**

| Modelo | Precision | Recall | F1 | PR-AUC | Alertas |
| --- | --- | --- | --- | --- | --- |
| One-Class SVM (nu=0.25) | 0.6258 | 0.3208 | 0.4241 | 0.6836 | 39.44% (306 VP / 183 FP) |
| LOF (n=20, contamination=0.15) | 0.7403 | 0.1405 | 0.2361 | 0.7402 | 14.60% (134 VP / 47 FP) |

**Recomendação:** o **SVM** para maximizar cobertura (F1 e recall ~2× o LOF; captura 100% de `falha_r06` e `falha_r09`, as regras com feature correspondente) e o **LOF** quando o custo de revisão prevalece (precision 0.74 alertando só 14,6% da base).

**Limitações conhecidas:** as falhas do rótulo vêm de apenas 4 regras (`r03` CEP, `r02` FK cliente, `r06` principal, `r09` logradouro); a cobertura concentra-se em r06/r09 (100%) e cai em r02 (39,7% SVM) e r03 (28,6% SVM / 11,8% LOF). O PR-AUC de ambos fica abaixo do baseline aleatório (954/1240 = 0.769) e a precision nos pontos de operação escolhidos (0.6258 no SVM, 0.7403 no LOF) também fica abaixo dele — ou seja, contra este rótulo fraco, os scores não agregam ordenação útil além de simplesmente alertar tudo (alertar os 1.240 daria recall 100% com precision 0.769).

**Caminhos futuros (não implementados):** features brutas de endereço (UF válida, apelido válido, número nulo, qtd. de endereços principais), rótulo fraco completo (somar `regra_NN_falhou` da Silver de clientes ao y_true) e Isolation Forest como terceiro modelo.